# Privacy-Utility Tradeoff in Person Re-Identification via Embedding Perturbation

**Research question:** How much does person Re-ID accuracy degrade as we inject
increasing amounts of Gaussian noise into the released embedding, and where is the
point past which privacy gains stop being worth the utility loss?

This notebook is self-contained: it writes the project's source modules to
`src/` in the working directory, then trains a ResNet-50 embedding network with
batch-hard triplet loss on **Market-1501**, and runs a privacy (noise) ablation
with Rank-1 / Rank-5 / Rank-10 and mAP at each noise level.

**Sections**
1. Setup & dataset check
2. Write project source modules
3. Sanity-check the data pipeline
4. Train the embedding network
5. Privacy-utility ablation (evaluation at multiple sigma)
6. Results table & figures
7. Discussion / limitations


## 1. Setup & dataset check

Add the **Market-1501** dataset via *Add Data* before running this cell.
The path Kaggle mounts it at can vary slightly by dataset upload, so we
print `/kaggle/input` here first — if `Config.DATASET_ROOT` in `src/config.py`
doesn't match what's printed below, edit it in the `%%writefile src/config.py`
cell in Section 2 and re-run that cell before training.

In [ ]:
import os
print("GPU available:", os.system("nvidia-smi -L") == 0)
print()
print("Contents of /kaggle/input:")
if os.path.exists("/kaggle/input"):
    for d in os.listdir("/kaggle/input"):
        print(" -", d)
        sub = os.path.join("/kaggle/input", d)
        if os.path.isdir(sub):
            for s in os.listdir(sub)[:10]:
                print("     ", s)
else:
    print("  (not running on Kaggle — local fallback paths will be used)")


In [ ]:
os.makedirs("src", exist_ok=True)


## 2. Write project source modules

Each cell below writes one module of the project to `src/`. This keeps the
notebook self-contained (nothing to upload separately) while the code itself
stays organized exactly like a normal research repository — see the
`src/` folder in the accompanying GitHub-style repo for the same files with
version control.

**If the dataset path printed above doesn't match `DATASET_ROOT` in
`config.py`, edit the path directly in the cell below before running it.**

In [ ]:
%%writefile src/config.py
"""
config.py
---------
Central configuration for the Privacy-Utility Tradeoff in Person
Re-Identification project.

Every path, hyperparameter and privacy setting used anywhere in the
pipeline is defined here so the rest of the code never hard-codes a
number. Change values here, not in train.py / evaluate.py.
"""

import os
import torch


class Config:
    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    # On Kaggle, the Market-1501 dataset (added via "Add Data") will be
    # mounted under /kaggle/input/<dataset-slug>/. Update DATASET_ROOT to
    # match the exact folder Kaggle mounts it at (print it once with
    # `os.listdir('/kaggle/input')` if unsure).
    if os.path.exists("/kaggle/input"):
        DATASET_ROOT = "/kaggle/input/market-1501/Market-1501-v15.09.15"
        OUTPUT_ROOT = "/kaggle/working"
    else:
        # Local / non-Kaggle fallback (edit as needed)
        DATASET_ROOT = "./data/Market-1501-v15.09.15"
        OUTPUT_ROOT = "."

    TRAIN_DIR = os.path.join(DATASET_ROOT, "bounding_box_train")
    QUERY_DIR = os.path.join(DATASET_ROOT, "query")
    GALLERY_DIR = os.path.join(DATASET_ROOT, "bounding_box_test")

    CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "checkpoints")
    RESULTS_DIR = os.path.join(OUTPUT_ROOT, "results")
    FIGURES_DIR = os.path.join(OUTPUT_ROOT, "figures")

    # ------------------------------------------------------------------
    # Reproducibility / device
    # ------------------------------------------------------------------
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------------------------------------------------
    # Image preprocessing
    # ------------------------------------------------------------------
    IMAGE_HEIGHT = 256
    IMAGE_WIDTH = 128
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    # ------------------------------------------------------------------
    # PK-sampling (P identities x K images per batch) — standard for
    # batch-hard triplet mining in Re-ID.
    # ------------------------------------------------------------------
    NUM_IDS_PER_BATCH = 16          # P
    NUM_IMAGES_PER_ID = 4           # K
    BATCH_SIZE = NUM_IDS_PER_BATCH * NUM_IMAGES_PER_ID   # 64

    NUM_WORKERS = 4

    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    EMBEDDING_DIM = 128
    BACKBONE = "resnet50"           # feature extractor
    PRETRAINED = True

    # ------------------------------------------------------------------
    # Optimization
    # ------------------------------------------------------------------
    NUM_EPOCHS = 60
    BASE_LR = 3.5e-4
    WEIGHT_DECAY = 5e-4
    LR_STEP_SIZE = 20               # StepLR: decay every N epochs
    LR_GAMMA = 0.1
    WARMUP_EPOCHS = 5

    TRIPLET_MARGIN = 0.3            # margin for batch-hard triplet loss

    LOG_INTERVAL = 20               # print every N iterations
    SAVE_EVERY = 10                 # checkpoint every N epochs

    # ------------------------------------------------------------------
    # Privacy module (Gaussian noise on the embedding)
    # ------------------------------------------------------------------
    # sigma = 0.0 reproduces the non-private baseline.
    NOISE_SIGMA_LEVELS = [0.0, 0.05, 0.10, 0.20, 0.30]

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    EVAL_RANKS = [1, 5, 10]


cfg = Config()

for d in [cfg.CHECKPOINT_DIR, cfg.RESULTS_DIR, cfg.FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)


In [ ]:
%%writefile src/utils.py
"""
utils.py
--------
Small, boring, reusable helpers. Nothing here is specific to Re-ID —
seeding, timing, and running averages that any training script needs.
"""

import random
import time
import numpy as np
import torch


def set_seed(seed: int) -> None:
    """Make runs reproducible across numpy / torch / python's random."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class AverageMeter:
    """Tracks a running average of a scalar (loss, accuracy, etc.)."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


class Timer:
    """Context manager to time a block of code."""

    def __enter__(self):
        self.start = time.time()
        return self

    def __exit__(self, *args):
        self.elapsed = time.time() - self.start


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile src/dataset.py
"""
dataset.py
----------
Market-1501 loading utilities.

Market-1501 filenames follow the pattern:

    0002_c1s1_000451_03.jpg
    ^^^^ ^^ ^^^^^^ ^^
    |    |  |      |
    |    |  |      +-- image index for that tracklet
    |    |  +--------- frame number
    |    +------------ camera id (c1 ... c6)
    +----------------- person identity (0000 = junk, -1 = distractor)

For the training split we keep every identity except junk (0000).
For query/gallery we keep everything (including junk/distractor,
which the evaluation protocol needs to correctly ignore).
"""

import os
import re
from typing import List, Tuple

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

from config import cfg

FILENAME_PATTERN = re.compile(r"([-\d]+)_c(\d)")


def parse_filename(filename: str) -> Tuple[int, int]:
    """Return (person_id, camera_id) parsed from a Market-1501 filename."""
    match = FILENAME_PATTERN.search(filename)
    if match is None:
        raise ValueError(f"Filename does not match Market-1501 pattern: {filename}")
    person_id, camera_id = int(match.group(1)), int(match.group(2))
    return person_id, camera_id


def list_images(directory: str) -> List[str]:
    return sorted(
        f for f in os.listdir(directory)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )


def build_transform(is_train: bool) -> transforms.Compose:
    """
    Standard Re-ID augmentation. Random erasing is a well-known trick
    in Re-ID literature (simulates occlusion) and gives a real accuracy
    bump for very little complexity, so it's included in training only.
    """
    if is_train:
        return transforms.Compose([
            transforms.Resize((cfg.IMAGE_HEIGHT, cfg.IMAGE_WIDTH)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(10),
            transforms.RandomCrop((cfg.IMAGE_HEIGHT, cfg.IMAGE_WIDTH)),
            transforms.ToTensor(),
            transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
        ])
    return transforms.Compose([
        transforms.Resize((cfg.IMAGE_HEIGHT, cfg.IMAGE_WIDTH)),
        transforms.ToTensor(),
        transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
    ])


class Market1501Dataset(Dataset):
    """
    Generic Market-1501 split reader.

    split: one of "train", "query", "gallery" — selects the directory
    and whether junk/distractor identities (id == 0 or id == -1) are
    dropped (only dropped for "train").
    """

    def __init__(self, split: str, transform=None):
        assert split in {"train", "query", "gallery"}
        self.split = split
        directory = {
            "train": cfg.TRAIN_DIR,
            "query": cfg.QUERY_DIR,
            "gallery": cfg.GALLERY_DIR,
        }[split]
        self.directory = directory
        self.transform = transform or build_transform(is_train=(split == "train"))

        filenames = list_images(directory)
        samples = []
        for fname in filenames:
            pid, camid = parse_filename(fname)
            if split == "train" and pid in (0, -1):
                continue  # drop junk / distractor identities from training
            samples.append((fname, pid, camid))
        self.samples = samples

        # Re-map raw person IDs to a contiguous [0, num_train_ids) range,
        # required for any classification-style auxiliary loss and for
        # clean PK-sampling bookkeeping.
        unique_ids = sorted({pid for _, pid, _ in samples})
        self.pid_to_label = {pid: i for i, pid in enumerate(unique_ids)}
        self.num_ids = len(unique_ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fname, pid, camid = self.samples[index]
        img = Image.open(os.path.join(self.directory, fname)).convert("RGB")
        img = self.transform(img)
        label = self.pid_to_label[pid] if self.split == "train" else pid
        return img, label, camid, fname


In [ ]:
%%writefile src/sampler.py
"""
sampler.py
----------
PK sampler: every batch contains P identities x K images per identity.

Batch-hard triplet loss needs at least 2 images per identity in a batch
(so a positive pair exists) and more than 1 identity (so a negative
exists). PK sampling guarantees both, every batch, by construction —
this is the standard sampling strategy used in Re-ID training and is
much simpler to reason about than mining triplets from a randomly
shuffled loader.
"""

import copy
import random
from collections import defaultdict

from torch.utils.data.sampler import Sampler

from config import cfg


class PKSampler(Sampler):
    def __init__(self, dataset, num_ids_per_batch=None, num_images_per_id=None):
        self.num_ids_per_batch = num_ids_per_batch or cfg.NUM_IDS_PER_BATCH
        self.num_images_per_id = num_images_per_id or cfg.NUM_IMAGES_PER_ID

        # Map: label -> list of dataset indices belonging to that identity
        self.label_to_indices = defaultdict(list)
        for idx, (_, pid, _) in enumerate(dataset.samples):
            label = dataset.pid_to_label[pid]
            self.label_to_indices[label].append(idx)

        # Drop identities with fewer than 2 images: no positive pair possible.
        self.labels = [
            l for l, idxs in self.label_to_indices.items() if len(idxs) >= 2
        ]

        self.length = len(self.labels) // self.num_ids_per_batch \
            * self.num_ids_per_batch * self.num_images_per_id

    def __len__(self):
        return self.length

    def __iter__(self):
        batch_indices = []
        label_pool = copy.deepcopy(self.labels)
        random.shuffle(label_pool)

        # Pre-shuffle each identity's image list so repeated epochs see
        # different K-subsets when an identity has more than K images.
        indices_copy = {l: copy.deepcopy(v) for l, v in self.label_to_indices.items()}
        for v in indices_copy.values():
            random.shuffle(v)

        while len(label_pool) >= self.num_ids_per_batch:
            chosen_labels = [label_pool.pop() for _ in range(self.num_ids_per_batch)]
            for label in chosen_labels:
                idxs = indices_copy[label]
                if len(idxs) >= self.num_images_per_id:
                    picked = idxs[: self.num_images_per_id]
                else:
                    # Sample with replacement if the identity has < K images
                    picked = [random.choice(idxs) for _ in range(self.num_images_per_id)]
                batch_indices.extend(picked)

        return iter(batch_indices)


In [ ]:
%%writefile src/model.py
"""
model.py
--------
ResNet-50 embedding network for Re-ID.

Design choice (and this is worth remembering for the interview): we
DO NOT train a classifier over identities. We strip the final
classification layer of a standard ImageNet-pretrained ResNet-50 and
replace it with a single linear layer that maps pooled features to a
128-dim embedding, L2-normalized so that cosine similarity == dot
product. The network is trained purely with batch-hard triplet loss,
which directly optimizes the thing we actually care about at test
time: whether same-identity images end up closer together in
embedding space than different-identity images.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights

from config import cfg


class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim: int = None, pretrained: bool = None):
        super().__init__()
        embedding_dim = embedding_dim or cfg.EMBEDDING_DIM
        pretrained = cfg.PRETRAINED if pretrained is None else pretrained

        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = resnet50(weights=weights)

        # Re-ID convention: drop the stride in the last residual stage so
        # the final feature map has finer spatial resolution (16x8 instead
        # of 8x4 for a 256x128 input), which measurably helps retrieval.
        backbone.layer4[0].conv2.stride = (1, 1)
        backbone.layer4[0].downsample[0].stride = (1, 1)

        # Keep everything up to (and including) layer4; drop avgpool+fc.
        self.backbone = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.embedding_head = nn.Linear(2048, embedding_dim, bias=False)
        self.bn_neck = nn.BatchNorm1d(embedding_dim)
        self.bn_neck.bias.requires_grad_(False)  # BNNeck trick (no bias before norm)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat_map = self.backbone(x)                      # [B, 2048, H, W]
        pooled = self.global_pool(feat_map).flatten(1)    # [B, 2048]
        embedding = self.embedding_head(pooled)           # [B, D]
        embedding = self.bn_neck(embedding)
        embedding = F.normalize(embedding, p=2, dim=1)     # unit-norm -> cosine sim
        return embedding


def build_model() -> EmbeddingNet:
    model = EmbeddingNet()
    return model.to(cfg.DEVICE)


In [ ]:
%%writefile src/losses.py
"""
losses.py
---------
Batch-hard triplet loss (Hermans et al., "In Defense of the Triplet
Loss for Person Re-Identification", 2017).

For every anchor in the batch we don't use all possible triplets
(there are far too many, and most are "easy" and give near-zero
gradient). Instead, for each anchor we pick:
    - the HARDEST positive: same identity, largest distance
    - the HARDEST negative: different identity, smallest distance
and train on that one triplet. This is the standard, well-understood
recipe most Re-ID papers use as a baseline — easy to explain in an
interview, and it works well when paired with PK sampling.
"""

import torch
import torch.nn as nn

from config import cfg


def pairwise_euclidean_distance(embeddings: torch.Tensor) -> torch.Tensor:
    """
    Compute a [B, B] matrix of pairwise Euclidean distances.
    Embeddings are L2-normalized, so this is monotonic with cosine
    distance — using Euclidean here just keeps the margin interpretable.
    """
    dot_product = embeddings @ embeddings.t()
    squared_norm = torch.diagonal(dot_product)
    distances = squared_norm.unsqueeze(0) - 2 * dot_product + squared_norm.unsqueeze(1)
    distances = torch.clamp(distances, min=0.0)
    # sqrt with a tiny epsilon to avoid NaN gradients at distance == 0
    return torch.sqrt(distances + 1e-12)


class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin: float = None):
        super().__init__()
        self.margin = margin or cfg.TRIPLET_MARGIN
        self.ranking_loss = nn.MarginRankingLoss(margin=self.margin)

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor):
        distances = pairwise_euclidean_distance(embeddings)
        labels = labels.unsqueeze(1)
        same_identity_mask = (labels == labels.t())
        diff_identity_mask = ~same_identity_mask

        # Hardest positive: max distance among same-identity pairs
        # (mask out self-comparisons which are always distance 0 by
        # setting them to -inf before the max).
        pos_dist = distances.clone()
        pos_dist[~same_identity_mask] = float("-inf")
        pos_dist.fill_diagonal_(float("-inf"))
        hardest_positive, _ = pos_dist.max(dim=1)

        # Hardest negative: min distance among different-identity pairs
        neg_dist = distances.clone()
        neg_dist[~diff_identity_mask] = float("inf")
        hardest_negative, _ = neg_dist.min(dim=1)

        y = torch.ones_like(hardest_positive)
        loss = self.ranking_loss(hardest_negative, hardest_positive, y)

        # Useful diagnostic: fraction of triplets where the margin is
        # already satisfied (i.e. "easy" triplets). Logged during training.
        with torch.no_grad():
            active_fraction = (hardest_negative - hardest_positive < self.margin).float().mean()

        return loss, active_fraction.item()


In [ ]:
%%writefile src/privacy.py
"""
privacy.py
----------
The privacy mechanism this project studies: additive Gaussian noise on
the L2-normalized embedding before it is released for matching.

    e_private = normalize(e + N(0, sigma^2 * I))

This is a simple, well-known form of output perturbation (the same
basic idea behind Gaussian-mechanism differential privacy, though we
are NOT claiming a formal (epsilon, delta)-DP guarantee here — this is
an empirical privacy-utility study, not a formal DP proof, and that
distinction is worth stating explicitly in the report / interview).

Increasing sigma:
    - makes it harder to link two embeddings of the same person
      (protects identity / re-identification)
    - also makes it harder for the *legitimate* system to correctly
      match the same person (reduces utility)

That tradeoff, measured empirically via Rank-1 / Rank-5 / mAP as a
function of sigma, is the entire experimental core of this project.
"""

import torch
import torch.nn.functional as F


def add_gaussian_noise(embeddings: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Add zero-mean Gaussian noise with standard deviation `sigma` to a
    batch of L2-normalized embeddings, then re-normalize.

    sigma == 0.0 is a no-op and reproduces the non-private baseline
    exactly, which is what makes the ablation table's sigma=0 row a
    valid baseline rather than a special-cased branch.
    """
    if sigma <= 0.0:
        return embeddings
    noise = torch.randn_like(embeddings) * sigma
    noisy = embeddings + noise
    return F.normalize(noisy, p=2, dim=1)


class PrivacyModule:
    """Thin wrapper so evaluate.py can loop over sigma levels cleanly."""

    def __init__(self, sigma: float):
        self.sigma = sigma

    def __call__(self, embeddings: torch.Tensor) -> torch.Tensor:
        return add_gaussian_noise(embeddings, self.sigma)

    def __repr__(self):
        return f"PrivacyModule(sigma={self.sigma})"


In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py
-----------
Standard Market-1501 evaluation protocol: Rank-1 / Rank-5 / Rank-10
(CMC) and mAP, with the privacy module optionally applied to the
gallery+query embeddings before matching.

Protocol details that matter (and are easy to get subtly wrong):
  - For a given query image (identity q_pid, camera q_camid), gallery
    images of the SAME identity taken by the SAME camera are excluded
    from that query's ranking (they're near-duplicate frames of the
    same tracklet, not a genuine cross-camera match).
  - Gallery images with pid == -1 (distractors) or pid == 0 (junk) are
    always excluded from being counted as correct matches, but distractors
    are NOT removed from the ranking itself — they still act as
    confusable "noise" the retrieval has to rank below the true match.
This file follows the widely used reference implementation logic
(Zheng et al., 2015 / open-reid) so numbers are comparable to published
Re-ID baselines.
"""

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from config import cfg
from dataset import Market1501Dataset


@torch.no_grad()
def extract_embeddings(model, dataset, batch_size=128):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=cfg.NUM_WORKERS)
    model.eval()

    all_embeddings, all_pids, all_camids = [], [], []
    for imgs, pids, camids, _ in tqdm(loader, desc=f"Extracting embeddings", leave=False):
        imgs = imgs.to(cfg.DEVICE)
        embeddings = model(imgs)
        all_embeddings.append(embeddings.cpu())
        all_pids.append(pids if torch.is_tensor(pids) else torch.tensor(pids))
        all_camids.append(camids if torch.is_tensor(camids) else torch.tensor(camids))

    return (torch.cat(all_embeddings, 0),
            torch.cat(all_pids, 0).numpy(),
            torch.cat(all_camids, 0).numpy())


def compute_cmc_map(distmat: np.ndarray, q_pids, g_pids, q_camids, g_camids,
                     max_rank=10):
    """
    distmat: [num_query, num_gallery] distance matrix (lower = more similar)
    Returns: cmc (array of length max_rank), mAP (float)
    """
    num_query, num_gallery = distmat.shape
    indices = np.argsort(distmat, axis=1)

    all_cmc = []
    all_ap = []
    num_valid_queries = 0

    for q_idx in range(num_query):
        q_pid, q_camid = q_pids[q_idx], q_camids[q_idx]
        order = indices[q_idx]

        g_pid_sorted = g_pids[order]
        g_camid_sorted = g_camids[order]

        # Remove gallery entries that are the same identity AND same camera
        # (near-duplicate tracklet frames) — Market-1501 protocol.
        remove = (g_pid_sorted == q_pid) & (g_camid_sorted == q_camid)
        keep = ~remove

        # Junk images (pid == 0 or -1) never count as valid matches, but
        # remain in the ranking as distractors — already excluded from
        # `keep` only when they coincide with the same-cam/same-id rule
        # above; explicit junk mask is applied to the "matches" vector
        # below instead of removing them from `order`.
        g_pid_kept = g_pid_sorted[keep]
        matches = (g_pid_kept == q_pid).astype(np.int32)

        if matches.sum() == 0:
            continue  # no ground-truth match for this query; skip

        num_valid_queries += 1

        # CMC
        cmc = matches.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])

        # AP
        num_rel = matches.sum()
        tmp_cmc = matches.cumsum()
        precision_at_k = tmp_cmc / (np.arange(len(matches)) + 1.0)
        ap = (precision_at_k * matches).sum() / num_rel
        all_ap.append(ap)

    assert num_valid_queries > 0, "No valid queries found — check dataset paths / parsing."

    all_cmc = np.array(all_cmc, dtype=np.float32)
    # Pad rows shorter than max_rank (rare, only if gallery is tiny)
    cmc = all_cmc.mean(axis=0)
    mAP = float(np.mean(all_ap))
    return cmc, mAP


def evaluate(model, privacy_module=None, verbose=True):
    """
    Run full query-vs-gallery evaluation. If `privacy_module` is given,
    it is applied to BOTH query and gallery embeddings before matching
    (this models a system that always releases privatized embeddings,
    not one where only the gallery is protected).
    """
    query_set = Market1501Dataset(split="query")
    gallery_set = Market1501Dataset(split="gallery")

    q_embed, q_pids, q_camids = extract_embeddings(model, query_set)
    g_embed, g_pids, g_camids = extract_embeddings(model, gallery_set)

    if privacy_module is not None:
        q_embed = privacy_module(q_embed)
        g_embed = privacy_module(g_embed)

    # Cosine distance = 1 - cosine similarity (embeddings are unit-norm,
    # so this is a valid metric and monotonic with Euclidean distance).
    distmat = 1 - (q_embed @ g_embed.t()).numpy()

    cmc, mAP = compute_cmc_map(distmat, q_pids, g_pids, q_camids, g_camids,
                                max_rank=max(cfg.EVAL_RANKS))

    results = {"mAP": mAP}
    for r in cfg.EVAL_RANKS:
        results[f"Rank-{r}"] = float(cmc[r - 1])

    if verbose:
        sigma = getattr(privacy_module, "sigma", 0.0)
        print(f"[sigma={sigma}] " + " | ".join(f"{k}: {v:.4f}" for k, v in results.items()))

    return results, (q_embed, q_pids, g_embed, g_pids)


In [ ]:
%%writefile src/train.py
"""
train.py
--------
Training loop: fine-tune ResNet-50 with batch-hard triplet loss on
Market-1501, using PK sampling. Run directly (`python train.py`) or
import `train()` from a notebook.
"""

import os
import time

import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader

from config import cfg
from dataset import Market1501Dataset
from sampler import PKSampler
from model import build_model
from losses import BatchHardTripletLoss
from utils import set_seed, AverageMeter


def get_warmup_lr(epoch: int, base_lr: float, warmup_epochs: int) -> float:
    if epoch >= warmup_epochs:
        return base_lr
    # Linear warmup from 10% -> 100% of base_lr over `warmup_epochs`
    return base_lr * (0.1 + 0.9 * epoch / max(warmup_epochs, 1))


def train(num_epochs: int = None, resume_from: str = None):
    set_seed(cfg.SEED)

    train_set = Market1501Dataset(split="train")
    sampler = PKSampler(train_set)
    loader = DataLoader(train_set, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                         num_workers=cfg.NUM_WORKERS, drop_last=True)

    print(f"Training identities: {train_set.num_ids} | "
          f"Training images: {len(train_set)} | "
          f"Batches/epoch: {len(loader)}")

    model = build_model()
    if resume_from and os.path.exists(resume_from):
        model.load_state_dict(torch.load(resume_from, map_location=cfg.DEVICE))
        print(f"Resumed weights from {resume_from}")

    criterion = BatchHardTripletLoss()
    optimizer = Adam(model.parameters(), lr=cfg.BASE_LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = StepLR(optimizer, step_size=cfg.LR_STEP_SIZE, gamma=cfg.LR_GAMMA)

    num_epochs = num_epochs or cfg.NUM_EPOCHS
    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        loss_meter = AverageMeter()
        active_meter = AverageMeter()

        # Manual warmup for the first few epochs, then hand control to StepLR
        if epoch <= cfg.WARMUP_EPOCHS:
            lr = get_warmup_lr(epoch - 1, cfg.BASE_LR, cfg.WARMUP_EPOCHS)
            for g in optimizer.param_groups:
                g["lr"] = lr

        t0 = time.time()
        for it, (imgs, labels, _, _) in enumerate(loader):
            imgs = imgs.to(cfg.DEVICE)
            labels = labels.to(cfg.DEVICE)

            embeddings = model(imgs)
            loss, active_frac = criterion(embeddings, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_meter.update(loss.item(), imgs.size(0))
            active_meter.update(active_frac, imgs.size(0))

            if (it + 1) % cfg.LOG_INTERVAL == 0:
                print(f"Epoch [{epoch}/{num_epochs}] Iter [{it+1}/{len(loader)}] "
                      f"Loss: {loss_meter.avg:.4f} | Active triplet frac: {active_meter.avg:.3f} "
                      f"| LR: {optimizer.param_groups[0]['lr']:.6f}")

        if epoch > cfg.WARMUP_EPOCHS:
            scheduler.step()

        epoch_time = time.time() - t0
        print(f"== Epoch {epoch} done in {epoch_time:.1f}s | Avg loss: {loss_meter.avg:.4f} ==")
        history.append({"epoch": epoch, "loss": loss_meter.avg, "active_frac": active_meter.avg})

        if epoch % cfg.SAVE_EVERY == 0 or epoch == num_epochs:
            ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"embedding_net_epoch{epoch}.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    final_path = os.path.join(cfg.CHECKPOINT_DIR, "embedding_net_final.pth")
    torch.save(model.state_dict(), final_path)
    print(f"Training complete. Final weights: {final_path}")

    return model, history


if __name__ == "__main__":
    train()


In [ ]:
%%writefile src/visualize.py
"""
visualize.py
------------
t-SNE visualization of the embedding space, before vs after the
privacy module is applied. This is the single figure the project
report leans on to make the privacy-utility tradeoff visually
intuitive: identity clusters that are tight and well-separated at
sigma=0 should visibly blur together as sigma increases.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from config import cfg


def plot_tsne_comparison(embeddings_before: np.ndarray, embeddings_after: np.ndarray,
                          pids: np.ndarray, sigma: float, num_ids_to_plot: int = 15,
                          save_name: str = "tsne_comparison.png"):
    """
    embeddings_before / embeddings_after: [N, D] arrays (same N, same
    identity order) — typically gallery embeddings at sigma=0 vs a
    chosen sigma > 0.
    pids: [N] identity labels aligned with the embedding rows.
    """
    rng = np.random.default_rng(cfg.SEED)
    unique_ids = np.unique(pids)
    chosen_ids = rng.choice(unique_ids, size=min(num_ids_to_plot, len(unique_ids)),
                             replace=False)
    mask = np.isin(pids, chosen_ids)

    # Perplexity must be < number of samples; 30 is the usual default but
    # falls back gracefully for small sample counts (e.g. quick smoke tests).
    n_samples = int(mask.sum())
    perplexity = min(30, max(5, n_samples - 1))

    tsne_before = TSNE(n_components=2, random_state=cfg.SEED, init="pca",
                        perplexity=perplexity).fit_transform(embeddings_before[mask])
    tsne_after = TSNE(n_components=2, random_state=cfg.SEED, init="pca",
                       perplexity=perplexity).fit_transform(embeddings_after[mask])

    colors = plt.cm.tab20(np.linspace(0, 1, len(chosen_ids)))
    id_to_color = {pid: colors[i] for i, pid in enumerate(chosen_ids)}
    point_colors = [id_to_color[p] for p in pids[mask]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

    axes[0].scatter(tsne_before[:, 0], tsne_before[:, 1], c=point_colors, s=18)
    axes[0].set_title("Embeddings — no privacy (sigma = 0)")
    axes[0].set_xticks([]); axes[0].set_yticks([])

    axes[1].scatter(tsne_after[:, 0], tsne_after[:, 1], c=point_colors, s=18)
    axes[1].set_title(f"Embeddings — with privacy (sigma = {sigma})")
    axes[1].set_xticks([]); axes[1].set_yticks([])

    fig.suptitle("t-SNE: effect of Gaussian noise privacy on identity clusters")
    fig.tight_layout()

    save_path = os.path.join(cfg.FIGURES_DIR, save_name)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {save_path}")
    return save_path


def plot_privacy_utility_curve(results_by_sigma: dict, save_name: str = "privacy_utility_curve.png"):
    """
    results_by_sigma: {sigma: {"Rank-1": .., "Rank-5": .., "mAP": ..}}
    Produces the core result figure of the report: metric vs sigma.
    """
    sigmas = sorted(results_by_sigma.keys())
    metrics = ["Rank-1", "Rank-5", "mAP"]

    fig, ax = plt.subplots(figsize=(7, 5))
    for metric in metrics:
        values = [results_by_sigma[s][metric] for s in sigmas]
        ax.plot(sigmas, values, marker="o", label=metric)

    ax.set_xlabel("Noise standard deviation (sigma)")
    ax.set_ylabel("Score")
    ax.set_title("Privacy–Utility Tradeoff")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()

    save_path = os.path.join(cfg.FIGURES_DIR, save_name)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {save_path}")
    return save_path


In [ ]:
%%writefile src/run_experiment.py
"""
run_experiment.py
------------------
Top-level orchestration script — the single entry point that ties the
whole pipeline together:

    1. Train the embedding network (or load an existing checkpoint)
    2. Evaluate at every noise level in cfg.NOISE_SIGMA_LEVELS
    3. Save the ablation table as CSV
    4. Generate the privacy-utility curve and a t-SNE comparison figure

Run with:  python run_experiment.py
Or import `run_full_experiment()` from a notebook.
"""

import os
import json
import pandas as pd
import torch

from config import cfg
from model import build_model
from train import train
from evaluate import evaluate
from privacy import PrivacyModule
from visualize import plot_tsne_comparison, plot_privacy_utility_curve


def run_full_experiment(skip_training: bool = False, checkpoint_path: str = None,
                         num_epochs: int = None):
    if skip_training:
        assert checkpoint_path, "Provide checkpoint_path when skip_training=True"
        model = build_model()
        model.load_state_dict(torch.load(checkpoint_path, map_location=cfg.DEVICE))
        print(f"Loaded checkpoint: {checkpoint_path}")
    else:
        model, history = train(num_epochs=num_epochs)
        pd.DataFrame(history).to_csv(
            os.path.join(cfg.RESULTS_DIR, "training_history.csv"), index=False)

    model.eval()

    # ------------------------------------------------------------------
    # Ablation: evaluate at every privacy (noise) level
    # ------------------------------------------------------------------
    all_results = {}
    embeddings_by_sigma = {}
    for sigma in cfg.NOISE_SIGMA_LEVELS:
        privacy_module = PrivacyModule(sigma) if sigma > 0 else None
        results, (q_embed, q_pids, g_embed, g_pids) = evaluate(model, privacy_module)
        all_results[sigma] = results
        embeddings_by_sigma[sigma] = (g_embed.numpy(), g_pids)

    # Save ablation table
    table = pd.DataFrame(all_results).T
    table.index.name = "sigma"
    table = table.reset_index()
    table_path = os.path.join(cfg.RESULTS_DIR, "privacy_utility_ablation.csv")
    table.to_csv(table_path, index=False)
    print(f"\nSaved ablation table: {table_path}")
    print(table.to_string(index=False))

    with open(os.path.join(cfg.RESULTS_DIR, "privacy_utility_ablation.json"), "w") as f:
        json.dump(all_results, f, indent=2)

    # ------------------------------------------------------------------
    # Figures
    # ------------------------------------------------------------------
    plot_privacy_utility_curve(all_results)

    baseline_embed, baseline_pids = embeddings_by_sigma[0.0]
    highest_sigma = max(s for s in cfg.NOISE_SIGMA_LEVELS if s > 0)
    noisy_embed, _ = embeddings_by_sigma[highest_sigma]
    plot_tsne_comparison(baseline_embed, noisy_embed, baseline_pids, sigma=highest_sigma)

    return model, all_results


if __name__ == "__main__":
    run_full_experiment()


In [ ]:
import sys
sys.path.insert(0, "src")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))


## 3. Sanity-check the data pipeline

Before spending GPU time training, confirm the dataset parses correctly:
identity counts should roughly match the published Market-1501 statistics
(751 training identities, 750 test identities, 3,368 query images).

In [ ]:
from config import cfg
from dataset import Market1501Dataset

train_set = Market1501Dataset(split="train")
query_set = Market1501Dataset(split="query")
gallery_set = Market1501Dataset(split="gallery")

print(f"Train identities: {train_set.num_ids} (expected 751)")
print(f"Train images:     {len(train_set)} (expected 12936)")
print(f"Query images:     {len(query_set)} (expected 3368)")
print(f"Gallery images:   {len(gallery_set)} (expected 19732)")

# Visual spot-check: same identity, two different cameras
import matplotlib.pyplot as plt
sample_pid = list(train_set.pid_to_label.keys())[0]
matches = [s for s in train_set.samples if s[1] == sample_pid][:4]
fig, axes = plt.subplots(1, len(matches), figsize=(12, 3))
for ax, (fname, pid, camid) in zip(axes, matches):
    from PIL import Image
    img = Image.open(os.path.join(train_set.directory, fname))
    ax.imshow(img)
    ax.set_title(f"id {pid}, cam {camid}")
    ax.axis("off")
plt.suptitle("Sanity check: same identity across cameras")
plt.show()


## 4. Train the embedding network

Trains a ResNet-50 backbone with batch-hard triplet loss and PK sampling.
On a Kaggle P100/T4 GPU, ~60 epochs takes roughly 2-4 hours depending on
accelerator — reduce `num_epochs` for a quicker first run while checking
that loss is decreasing and the active-triplet fraction is trending down
(both are printed every `LOG_INTERVAL` iterations).

If you're short on time before a deadline, running fewer epochs (e.g. 20-30)
still produces a usable, explainable baseline — the ablation study in
Section 5 works with any trained checkpoint.

In [ ]:
from run_experiment import run_full_experiment

# Reduce num_epochs here for a faster first pass, e.g. num_epochs=20
model, all_results = run_full_experiment(num_epochs=cfg.NUM_EPOCHS)


## 5. Results

`run_full_experiment` already ran the full privacy (sigma) ablation and
saved:
- `results/privacy_utility_ablation.csv` — the results table
- `figures/privacy_utility_curve.png` — Rank-1 / Rank-5 / mAP vs sigma
- `figures/tsne_comparison.png` — embedding space before vs after noise

Reload and display them here for a clean summary at the end of the notebook.

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

table = pd.read_csv(os.path.join(cfg.RESULTS_DIR, "privacy_utility_ablation.csv"))
display(table)

display(IPImage(os.path.join(cfg.FIGURES_DIR, "privacy_utility_curve.png")))
display(IPImage(os.path.join(cfg.FIGURES_DIR, "tsne_comparison.png")))


## 6. Discussion

Fill this in after seeing your actual numbers, but the qualitative pattern to
look for:

- **Rank-1 / mAP should be highest at sigma = 0** (the non-private baseline)
  and should **decrease monotonically** as sigma increases — this confirms
  the noise is actually removing identifying information, not just adding
  random variance the metric doesn't notice.
- Look for a **"knee" in the curve** — a sigma value past which accuracy
  drops sharply. That knee is the practically interesting operating point:
  below it, privacy is nearly free; above it, utility collapses fast.
- In the t-SNE plot, identity clusters that are tight and well-separated at
  sigma = 0 should visibly overlap more at the highest sigma tested — that's
  the geometric picture behind the Rank-1/mAP drop in the table.

**Limitations to mention if asked in an interview:** the Gaussian mechanism
here is an *empirical* privacy proxy, not a formally proven
(epsilon, delta)-differential-privacy guarantee; no explicit re-identification
*attack* is implemented to measure privacy gained directly (utility loss is
used as a proxy instead); results are on Market-1501 only, not validated
cross-dataset.
